# Gemma 4 typed evidence repair - Kaggle setup and smoke tests

This notebook establishes the **P2 tqdm red/green bug test** and a **Gemma 4 E4B Q4_0 GPU inference check**. It does not yet implement agent conditions A/B/C or generate research results. Use a Kaggle notebook with **T4 x2**, Internet enabled for setup, this project ZIP attached as input, and an official E4B QAT Q4_0 GGUF attached as a model input. Save a notebook version after each checkpoint. See `protocol-v1.1.md`.


In [ ]:
import os, pathlib, subprocess, sys, json, time, shutil, glob
WORK = pathlib.Path('/kaggle/working')
print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=True)
print('Free disk:', shutil.disk_usage(WORK).free // (1024**3), 'GiB')


In [ ]:
# Select the uploaded project archive. If several ZIPs are attached, set PROJECT_ZIP explicitly.
PROJECT_ZIP = ''
candidates = sorted(pathlib.Path('/kaggle/input').rglob('Gemma4-Typed-Evidence-Repair-Project.zip'))
if not PROJECT_ZIP:
    if len(candidates) != 1:
        raise RuntimeError(f'Found {len(candidates)} project ZIPs; set PROJECT_ZIP to the attached file: {candidates}')
    PROJECT_ZIP = str(candidates[0])
subprocess.run(['unzip', '-q', '-o', PROJECT_ZIP, '-d', str(WORK)], check=True)
PROJECT = WORK / 'gemma4-typed-evidence-repair'
assert (PROJECT / 'cases.json').is_file(), PROJECT
print('Project:', PROJECT)


In [ ]:
# Install an isolated Python 3.8 environment for the historical tqdm test.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.8'], check=True)
VENV = WORK / 'pilot-env38'
subprocess.run(['uv', 'venv', '--python', '3.8', str(VENV)], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', str(VENV / 'bin/python'),
                '-r', str(PROJECT / 'requirements-tqdm-pilot.txt')], check=True)
print(subprocess.check_output([str(VENV / 'bin/python'), '--version'], text=True).strip())


In [ ]:
# P2: check that the regression test fails on the old commit and passes on the fix.
OUT = WORK / 'gemma4-repair-results'
OUT.mkdir(exist_ok=True)
check_dir = OUT / 'P2-red-green'
if check_dir.exists():
    print('Previous evidence retained:', check_dir / 'result.json')
else:
    subprocess.run([sys.executable, str(PROJECT / 'scripts/reproduce.py'), '--case', 'P2',
                    '--python', str(VENV / 'bin/python'), '--output', str(check_dir)], check=True)
print(json.loads((check_dir / 'result.json').read_text())['red_green'])


## Attach the model and build the inference server

Attach Google's `gemma-4-E4B-it-qat-q4_0-gguf` as a Kaggle model input. The cell below **does not download weights**; it selects one attached `.gguf` file. The next cell builds the official `llama.cpp` repository with CUDA support and records its exact commit. Its first build can take time. Use one GPU to keep memory and timing consistent.


In [ ]:
MODEL_GGUF = ''
found = sorted(pathlib.Path('/kaggle/input').rglob('*.gguf'))
if not MODEL_GGUF:
    if len(found) != 1:
        raise RuntimeError(f'Found {len(found)} GGUF files. Attach the official model and set MODEL_GGUF: {found[:12]}')
    MODEL_GGUF = str(found[0])
assert pathlib.Path(MODEL_GGUF).is_file()
print('Model file:', MODEL_GGUF, 'GiB:', round(pathlib.Path(MODEL_GGUF).stat().st_size/1024**3, 2))


In [ ]:
LLAMA_SRC = WORK / 'llama.cpp'
LLAMA_BUILD = LLAMA_SRC / 'build'
if not LLAMA_SRC.exists():
    subprocess.run(['git', 'clone', '--quiet', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_SRC)], check=True)
commit = subprocess.check_output(['git', '-C', str(LLAMA_SRC), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['cmake', '-S', str(LLAMA_SRC), '-B', str(LLAMA_BUILD),
                '-DGGML_CUDA=ON', '-DCMAKE_CUDA_ARCHITECTURES=75',
                '-DCMAKE_BUILD_TYPE=Release', '-DLLAMA_BUILD_SERVER=ON'], check=True)
subprocess.run(['cmake', '--build', str(LLAMA_BUILD), '--config', 'Release', '-j', '4'], check=True)
(OUT / 'inference-build.json').write_text(json.dumps({'llama_cpp_commit':commit,'cuda_architecture':'75','model_path':MODEL_GGUF}, indent=2))
print('llama.cpp commit:', commit)


In [ ]:
# Start the local server on GPU 0 only. Rerunning this cell reuses an existing healthy server.
import urllib.request
PORT = 8080
server_log = OUT / 'llama-server.log'
try:
    urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=2)
    print('Server already healthy')
except Exception:
    command = [str(LLAMA_BUILD / 'bin/llama-server'), '-m', MODEL_GGUF,
               '--host', '127.0.0.1', '--port', str(PORT), '--ctx-size', '16384',
               '--n-gpu-layers', '99', '--parallel', '1']
    log_stream = open(server_log, 'a')
    environment = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
    SERVER = subprocess.Popen(command, stdout=log_stream, stderr=subprocess.STDOUT, env=environment)
    for attempt in range(180):
        if SERVER.poll() is not None:
            raise RuntimeError('Server exited; inspect ' + str(server_log) + '\n' + server_log.read_text()[-3000:])
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=2)
            print('Server ready on GPU 0'); break
        except Exception:
            time.sleep(2)
    else:
        raise TimeoutError('Server load timed out; inspect ' + str(server_log))


In [ ]:
# One inference probe. Its result and model hash are written to the output folder.
probe = OUT / 'model-smoke.json'
if probe.exists():
    print('Previous evidence retained:', probe)
else:
    subprocess.run([sys.executable, str(PROJECT / 'scripts/model_smoke.py'),
                    '--model-file', MODEL_GGUF, '--output', str(probe)], check=True)
print('Saved files:', [p.name for p in OUT.iterdir() if p.is_file()])


## Checkpoint

Save the notebook version and download `/kaggle/working/gemma4-repair-results` as a ZIP if needed. `P2-red-green/result.json` proves only issue reproducibility. `model-smoke.json` proves only model inference. **Neither is an agent repair result.** Next we will add the same controlled agent harness for conditions A/B/C and freeze the 12-case evaluation set before using it.
